# Urban Heat — Shenzhen / Shanghai / Beijing (500m grid, 2018-2024 summer)

Pipeline overview (matches the plan we agreed on):

1. Pull each city's municipal administrative boundary from Aliyun DataV.GeoAtlas and dissolve it into a
   single city outline.
2. Build a regular 500m x 500m grid for each city in its own local UTM projection, clipped to the city
   boundary. Grid IDs are `city_row_col`; each cell centroid also gets a geohash6 label purely as a
   human-readable location tag (it does not define the cell size).
3. Landsat 8+9 Collection 2 Level-2, scale factors + QA_PIXEL/QA_RADSAT masking (same as Lab2/3), mean
   summer (Jun-Aug) composite for each year 2018-2024, with LST (Celsius), NDVI, NDBI computed.
4. For each city x year, `reduceRegions` aggregates every grid cell's mean values and batch-exports the
   result (default: EE Asset, so it doesn't eat into your almost-full Google Drive quota; a Drive fallback
   is included too).
5. Locally merge everything into one table and add an `LST_anomaly_C` column (cell LST minus that city's
   own mean for that year), so cross-city comparisons aren't dominated by background climate differences
   (e.g. Shenzhen being inherently warmer than Beijing).
6. Basic visualization: a single-city gridded heat map and a cross-city year-over-year trend comparison.

**Packages needed to run this** (your `houpu_py` env should already have `ee`/`geemap`/`pandas`;
`geopandas`/`shapely`/`pyproj`/`requests` were also used in Lab4, so you likely have them too — if not:
`pip install geopandas shapely pyproj requests`).


In [1]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import json
import os
import time
import matplotlib.pyplot as plt
from shapely.geometry import box

ee.Authenticate()
ee.Initialize(project='houpuprj')


/opt/anaconda3/envs/houpu_py/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/opt/anaconda3/envs/houpu_py/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/opt/anaconda3/envs/houpu_py/lib/python3.9/site-packages/google/api_core/_python_version_support.py:234: FutureWarning: You are using a non-supported Python version (3.9.19). Google will not post 

### Note: a safer geopandas -> ee.FeatureCollection helper

`geemap.geopandas_to_ee` writes a temporary GeoJSON file into the current working directory before
loading it into Earth Engine. If the notebook's kernel happens to have a read-only working directory
(e.g. `cwd == '/'`), that write fails with `Read-only file system` even though nothing else is wrong.
`gdf_to_ee_fc` below does the same conversion entirely in memory (no temp file), so it works regardless
of what the kernel's cwd is. All the `geemap.geopandas_to_ee(...)` calls later in this notebook have been
switched to use it instead.


In [2]:
def gdf_to_ee_fc(gdf, geodesic=True):
    gdf_wgs = gdf if (gdf.crs is None or gdf.crs.to_epsg() == 4326) else gdf.to_crs(epsg=4326)
    return geemap.geojson_to_ee(json.loads(gdf_wgs.to_json()), geodesic=geodesic)


## Step 0: Configuration

In [3]:
CITIES = {
    "Shenzhen": "440300",
    "Shanghai": "310000",
    "Beijing":  "110000",
}

YEARS = list(range(2018, 2025))     # 2018-2024
SUMMER_START_MD = "06-01"
SUMMER_END_MD   = "09-01"           # half-open interval, i.e. June-August

CELL_SIZE_M = 500                   # target grid cell size in meters

LOCAL_DATA_DIR = "/Users/houpuli/Library/CloudStorage/Dropbox-RedliningLab/HOUPU LI/UNDP_PRO/data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Path used for EE Asset exports. Before running Step 4, create a folder named
# urban_heat_grid under your houpuprj project in the GEE Assets panel
# (or via ee.data.createFolder, see the comment in Step 4).
ASSET_FOLDER = "projects/houpuprj/assets/urban_heat_grid"


## Step 1: Fetch the three city boundaries

`bound/{adcode}.json` sometimes returns a FeatureCollection split by the next admin level (districts),
and sometimes already returns the whole outline as one feature. `unary_union` folds whichever shape comes
back into a single city outline, so both cases are handled the same way.

Usage mirrors Lab4's Woolsey Fire GeoJSON -> `geemap.geopandas_to_ee` workflow, just going through the `gdf_to_ee_fc` helper defined above instead of calling `geemap.geopandas_to_ee` directly.


In [4]:
def get_city_boundary(adcode):
    url = f"https://geo.datav.aliyun.com/areas_v3/bound/{adcode}.json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    gj = resp.json()
    gdf = gpd.GeoDataFrame.from_features(gj["features"], crs="EPSG:4326")
    dissolved = gdf.geometry.unary_union
    return gpd.GeoDataFrame({"adcode": [adcode]}, geometry=[dissolved], crs="EPSG:4326")


city_boundaries = {}
for name, code_ in CITIES.items():
    gdf = get_city_boundary(code_)
    gdf.to_file(f"{LOCAL_DATA_DIR}/{name}_boundary.geojson", driver="GeoJSON")
    city_boundaries[name] = gdf
    area_km2 = gdf.to_crs(gdf.estimate_utm_crs()).area.iloc[0] / 1e6
    print(f"{name}: area = {area_km2:,.1f} km2, bounds = {gdf.total_bounds}")


Shenzhen: area = 2,138.2 km2, bounds = [113.751453  22.396343 114.628466  22.861749]
Shanghai: area = 7,986.3 km2, bounds = [120.856804  30.675593 122.247149  31.872716]
Beijing: area = 16,400.3 km2, bounds = [115.423411  39.442758 117.514583  41.0608  ]


In [5]:
# Visual sanity check that all three boundaries look right
m = geemap.Map()
for name, gdf in city_boundaries.items():
    ee_fc = gdf_to_ee_fc(gdf)
    m.addLayer(ee_fc, {}, f"{name} boundary")
m.centerObject(gdf_to_ee_fc(city_boundaries["Shanghai"]), 5)
m


Map(center=[31.209209844410502, 121.48840699513559], controls=(WidgetControl(options=['position', 'transparent…

## Step 2: Build a 500m x 500m grid for each city

Each city is gridded in its own local UTM zone based on its centroid (Shenzhen is near UTM 49N/50N,
Shanghai UTM 51N, Beijing UTM 50N), cut into clean 500m squares in that projected CRS, then clipped to
the city boundary (slivers left over from clipping near the boundary edge, area < 1 sq m, are dropped).

`geohash6` is just a readable label on each cell centroid — it has nothing to do with the actual cell
size (a real geohash6 cell covers roughly 1.22km x 0.61km, larger than this 500m grid; it's used here
purely as a place name). The geohash function has been checked against the standard Wikipedia test
vector (`geohash_encode(57.64911, 10.40744, 6) == 'u4pruy'`).


In [ ]:
_BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'

def geohash_encode(lat, lon, precision=6):
    lat_range, lon_range = [-90.0, 90.0], [-180.0, 180.0]
    geohash, bits, bit, ch, even = [], [16, 8, 4, 2, 1], 0, 0, True
    while len(geohash) < precision:
        if even:
            mid = (lon_range[0] + lon_range[1]) / 2
            if lon > mid:
                ch |= bits[bit]; lon_range[0] = mid
            else:
                lon_range[1] = mid
        else:
            mid = (lat_range[0] + lat_range[1]) / 2
            if lat > mid:
                ch |= bits[bit]; lat_range[0] = mid
            else:
                lat_range[1] = mid
        even = not even
        if bit < 4:
            bit += 1
        else:
            geohash.append(_BASE32[ch]); bit = 0; ch = 0
    return ''.join(geohash)


def get_utm_epsg(lon, lat):
    zone = int((lon + 180) / 6) + 1
    return 32600 + zone if lat >= 0 else 32700 + zone


def make_grid_for_city(city_gdf, city_name, cell_size=CELL_SIZE_M):
    lon0, lat0 = city_gdf.geometry.iloc[0].centroid.x, city_gdf.geometry.iloc[0].centroid.y
    utm_epsg = get_utm_epsg(lon0, lat0)

    city_utm = city_gdf.to_crs(epsg=utm_epsg)
    boundary = city_utm.geometry.iloc[0]
    minx, miny, maxx, maxy = boundary.bounds

    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)

    rows = []
    for i, x0 in enumerate(xs):
        for j, y0 in enumerate(ys):
            cell = box(x0, y0, x0 + cell_size, y0 + cell_size)
            if not cell.intersects(boundary):
                continue
            clipped = cell.intersection(boundary)
            if clipped.is_empty or clipped.area < 1.0:
                continue
            rows.append({"row": i, "col": j, "geometry": clipped})

    grid = gpd.GeoDataFrame(rows, crs=f"EPSG:{utm_epsg}")
    grid["grid_id"] = [f"{city_name}_{r}_{c}" for r, c in zip(grid["row"], grid["col"])]
    grid["cell_area_km2"] = grid.geometry.area / 1e6

    centroids_utm = grid.geometry.centroid
    centroids_wgs = gpd.GeoSeries(centroids_utm, crs=f"EPSG:{utm_epsg}").to_crs(epsg=4326)
    grid["lon"] = centroids_wgs.x.values
    grid["lat"] = centroids_wgs.y.values
    grid["geohash6"] = [geohash_encode(la, lo, 6) for la, lo in zip(grid["lat"], grid["lon"])]

    return grid.to_crs(epsg=4326), utm_epsg


city_grids = {}
for name, gdf in city_boundaries.items():
    grid, epsg = make_grid_for_city(gdf, name.lower())
    grid.to_file(f"{LOCAL_DATA_DIR}/{name}_grid_500m.geojson", driver="GeoJSON")
    city_grids[name] = grid
    print(f"{name}: {len(grid):,} cells (UTM EPSG:{epsg}), "
          f"mean cell area = {grid['cell_area_km2'].mean():.4f} km2")


## Step 3: Landsat 8/9 preprocessing + LST/NDVI/NDBI

Same `apply_scale_factors` / QA mask as Lab2/3, plus converting `ST_B10` to Celsius and adding NDVI/NDBI
bands so later analysis can relate the heat island signal to vegetation/built-up density. The two
satellite collections are simply `.merge()`d; for years before Landsat 9 launched (2021), the `l9` side
of the merge is just an empty collection, which `merge` handles fine.


In [ ]:
def apply_scale_factors(image):
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(optical, None, True).addBands(thermal, None, True)


def mask_landsat_qa(image):
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)
    return image.updateMask(qa_mask).updateMask(saturation_mask)


def preprocess_landsat(image):
    return mask_landsat_qa(apply_scale_factors(image))


def add_indices(image):
    lst_c = image.select('ST_B10').subtract(273.15).rename('LST_C')
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    ndbi = image.expression(
        '(SWIR - NIR) / (SWIR + NIR)',
        {'SWIR': image.select('SR_B6'), 'NIR': image.select('SR_B5')}
    ).rename('NDBI')
    return image.addBands([lst_c, ndvi, ndbi])


def get_annual_composite(geom, year):
    start, end = f"{year}-{SUMMER_START_MD}", f"{year}-{SUMMER_END_MD}"
    l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    dataset = (l8.merge(l9)
               .filterBounds(geom)
               .filterDate(start, end)
               .map(preprocess_landsat))
    composite = dataset.mean().clip(geom)
    return add_indices(composite)


In [ ]:
# Spot-check one city x year before running the full batch, to confirm the LST values look reasonable
check_city, check_year = "Shenzhen", 2023
geom_check = gdf_to_ee_fc(city_boundaries[check_city]).geometry()
composite_check = get_annual_composite(geom_check, check_year)

vis_lst = {'bands': ['LST_C'], 'min': 20, 'max': 45,
           'palette': ['313695', '4575b4', 'abd9e9', 'ffffbf', 'fdae61', 'd73027']}

m2 = geemap.Map()
m2.centerObject(geom_check, 10)
m2.addLayer(composite_check, vis_lst, f'{check_city} {check_year} LST (C)')
m2


## Step 4: Batch grid statistics + export

3 cities x 7 years = 21 tasks, each a `reduceRegions` call (Beijing alone has 60k+ cells in a given
year). A direct `getInfo()` would very likely time out or hit request-size limits, so this goes through
batch export.

Since you mentioned in Lab4 that your school Google Drive only has 1GB left, this defaults to exporting
to an **EE Asset** (no Drive usage at all) and reads it back with `ee_to_df` afterwards. If you'd rather
have local CSVs directly, swap in the commented-out `Export.table.toDrive` call instead (use one or the
other, not both).

**Before running this**: go to https://code.earthengine.google.com/ , open the Assets panel, and create a
folder named `urban_heat_grid` under your `houpuprj` project (matching `ASSET_FOLDER` above) — otherwise
the export will fail with a "path does not exist" error.


In [ ]:
tasks = {}
for city_name, boundary_gdf in city_boundaries.items():
    geom = gdf_to_ee_fc(boundary_gdf).geometry()
    grid_fc = gdf_to_ee_fc(
        city_grids[city_name][["grid_id", "geohash6", "lon", "lat", "geometry"]]
    )
    for year in YEARS:
        composite = get_annual_composite(geom, year)
        stats_fc = composite.select(['LST_C', 'NDVI', 'NDBI']).reduceRegions(
            collection=grid_fc,
            reducer=ee.Reducer.mean(),
            scale=30,
            tileScale=4
        )
        stats_fc = stats_fc.map(lambda f: f.set({'city': city_name, 'year': year}))

        description = f"{city_name}_{year}_grid_stats"

        task = ee.batch.Export.table.toAsset(
            collection=stats_fc,
            description=description,
            assetId=f"{ASSET_FOLDER}/{description}",
        )
        # Alternative: export CSV to Drive instead of an Asset — uncomment below and
        # comment out the toAsset call above instead
        # task = ee.batch.Export.table.toDrive(
        #     collection=stats_fc,
        #     description=description,
        #     folder="urban_heat_grid_exports",
        #     fileNamePrefix=description,
        #     fileFormat='CSV'
        # )

        task.start()
        tasks[description] = task
        print(f"submitted: {description}")


In [ ]:
def check_tasks(tasks, poll_seconds=30):
    """Poll task status until everything finishes. You can also skip this and just
    watch progress at https://code.earthengine.google.com/tasks ."""
    pending = dict(tasks)
    while pending:
        for desc, task in list(pending.items()):
            state = task.status()['state']
            if state in ('COMPLETED', 'FAILED', 'CANCELLED'):
                print(f"{desc}: {state}")
                pending.pop(desc)
        if pending:
            time.sleep(poll_seconds)

# check_tasks(tasks)   # uncomment to block until all 21 tasks finish; this can take a while


## Step 5: Read results back and merge into one table

Run this once every task shows COMPLETED in the Assets panel from Step 4.


In [ ]:
all_dfs = []
for city_name in CITIES:
    for year in YEARS:
        asset_id = f"{ASSET_FOLDER}/{city_name}_{year}_grid_stats"
        try:
            fc = ee.FeatureCollection(asset_id)
            df = geemap.ee_to_df(fc)
            df["city"] = city_name
            df["year"] = year
            all_dfs.append(df)
            print(f"loaded {city_name} {year}: {len(df)} rows")
        except Exception as e:
            print(f"skip {city_name} {year}: {e}")

master_df = pd.concat(all_dfs, ignore_index=True)
master_df = master_df.drop(columns=[c for c in ['system:index', '.geo'] if c in master_df.columns])

# Use each city's own mean for that year as the baseline, so raw climate differences
# between cities (e.g. subtropical Shenzhen vs temperate Beijing) don't swamp the
# within-city heat island signal.
city_year_mean = master_df.groupby(['city', 'year'])['LST_C'].transform('mean')
master_df['LST_anomaly_C'] = master_df['LST_C'] - city_year_mean

master_df.to_csv(f"{LOCAL_DATA_DIR}/urban_heat_grid_master.csv", index=False)
master_df.head()


## Step 6: Basic visualization

In [ ]:
plot_city, plot_year = "Shenzhen", 2023
sub = master_df[(master_df.city == plot_city) & (master_df.year == plot_year)]

fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(sub['lon'], sub['lat'], c=sub['LST_C'], cmap='RdYlBu_r', s=4)
plt.colorbar(sc, label='LST (°C)')
ax.set_title(f'{plot_city} {plot_year} Summer LST (500m grid)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()


In [ ]:
trend = master_df.groupby(['city', 'year'])[['LST_C', 'LST_anomaly_C']].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
for city_name in CITIES:
    d = trend[trend.city == city_name]
    ax.plot(d['year'], d['LST_C'], 'o-', label=city_name)
ax.set_xlabel('Year'); ax.set_ylabel('Mean summer LST (°C)')
ax.set_title('City-wide mean summer LST, 2018-2024')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.show()


## Ideas for extending this

- Scatter `LST_anomaly_C` against `NDVI`/`NDBI` and fit a regression line to quantify how much
  vegetation/built-up density explains local heat island intensity (same scatter + best-fit-line
  approach as NDWI vs DTM in Lab6).
- Use an actual rural reference area (rather than the city-wide mean) as the SUHI baseline — e.g. the
  mean LST over a ring of non-built-up land just outside each city — to match the standard SUHI
  intensity definition used in the literature more closely.
- Run Getis-Ord Gi* hot-spot analysis (`esda`/`pysal`) to find statistically significant heat cores
  within each city, rather than just looking at raw temperature.
- Break results down by district/county instead of the whole municipality — Beijing's mountainous
  outer districts especially will look very different from the urban core, and a city-wide mean can
  wash that out.
